# 扫雷：由数字提示推断出哪些格子安全

## 这一章讲什么

扫雷这个游戏从头到尾都在推理：看着已经翻开的格子上的数字，推断地雷藏在什么地方。这一章把这件工作整个交给 Z3，由它替我们逐格判断，下面先列出要点。

- 每一格都是一个 `Bool` 变量，它只回答一个问题：这里有没有地雷
- 数字格所说的“周围数字等于周围地雷数”，原样抄成约束
- 再用反证法问两遍：假设有雷会不会矛盾，假设没有雷会不会矛盾，靠 `push()` / `check()` / `pop()` 走完一轮

人在盘面上用手指一格一格试探，Z3 则把每一格都试了一遍。

## 扫雷的规则

盘面上写着的数字，意思很直接：这一格周围 8 格里埋了几颗地雷。数字格本身已经翻开，是安全的；`.` 则是还没有动过的格子，需要由程序替我们判断。

我们要写的程序只做一件事：拿到一个盘面，把所有可以断定不是地雷的格子挑出来，剩下的格子仍然留待猜测。

## 解题思路

一格一格来，每一格都试两种假设。

1. 先假设“这一格有地雷”，看它与已有的约束是否冲突
   - 冲突(`unsat`)→ 那就不可能真有雷 → 安全格
2. 再假设“这一格没有地雷”，同样看是否冲突
   - 冲突(`unsat`)→ 反过来，地雷肯定在这里 → 地雷格
3. 两种假设都活得下去 → 这一格现在还定不了(运气差的话，翻开就是雷)

这个套路称为反证法，其思路是：假设放进去，矛盾冒出来，就说明假设本身错了。

注意，麻烦的是假设之间不能互相污染。临时加上的假设，要用 `solver.push()`(先把当前状态保存起来)与 `solver.pop()`(用完退回保存点)夹在中间，这样处理下一格时，手上仍然是干净的约束。

## 代码

代码分成三个函数，各管一件事。

- `get_neighbors()` … 给出一个坐标，把周围 8 格列出来
- `solve_board()` … 真正去解盘面，交出安全格与地雷格两份名单
- `show_result()` … 把结果照盘面的样子画回去

In [1]:
from z3 import *

# 8 个方向（上下左右＋斜向）
DIRECTIONS = [
    (-1, -1), (-1, 0), (-1, 1),
    (0, -1),           (0, 1),
    (1, -1),  (1, 0),  (1, 1),
]


def get_neighbors(x, y, h, w):
    """返回格子 (x, y) 周围 8 格中位于盘面内的那些"""
    neighbors = []
    for dx, dy in DIRECTIONS:
        nx, ny = x + dx, y + dy
        if 0 <= nx < w and 0 <= ny < h:
            neighbors.append((nx, ny))
    return neighbors


def solve_board(board_text):
    """求解盘面，返回 (安全格, 地雷格)"""
    rows = board_text.strip().split()
    h, w = len(rows), len(rows[0])

    # 给数字格周围的未翻开格子建一个表示“有地雷”的 Bool 变量
    mine = {}
    for y, row in enumerate(rows):
        for x, v in enumerate(row):
            if v == '.':
                continue                      # 未翻开的格子本身跳过
            for nx, ny in get_neighbors(x, y, h, w):
                if rows[ny][nx] == '.':
                    mine[nx, ny] = Bool(f'mine_{nx}_{ny}')

    # 约束条件：数字格周围为 True 的变量个数 = 这个数字
    solver = Solver()
    for y, row in enumerate(rows):
        for x, v in enumerate(row):
            if v == '.':
                continue
            around = [mine[loc] for loc in get_neighbors(x, y, h, w) if loc in mine]
            if around:
                solver.add(Sum(around) == int(v))

    # 给每个格子加上假设，看是否矛盾
    safe, mines = [], []
    for (x, y), var in mine.items():
        solver.push()
        solver.add(var)                   # 假设：这里有地雷
        if solver.check() == unsat:       # 矛盾 → 不是地雷
            safe.append((x, y))
        solver.pop()

        solver.push()
        solver.add(Not(var))              # 假设：这里没有地雷
        if solver.check() == unsat:       # 矛盾 → 确定是地雷
            mines.append((x, y))
        solver.pop()

    return safe, mines


def show_result(board_text, safe, mines):
    """“#”是安全格，“*”是地雷格"""
    board = [list(row) for row in board_text.strip().split()]
    for x, y in mines:
        board[y][x] = '*'
    for x, y in safe:
        board[y][x] = '#'
    print("\n".join(" ".join(row) for row in board))

## 执行示例

![minesweeper](images/minesweeper01.png)

下面这个盘面我们不动手，交给代码去推断，看它能否把安全格与地雷格都定下来。

In [2]:
board_str = """
...201..................
...201..................
...1013.................
...1102.................
....1011123.............
..21100012...22211122...
..2000012.33212.3111....
..421002.310002.3.122...
....3224.31000112111....
........4.20000001121...
.......43.20000012.11...
.......3122211001.212...
.......202.3.10011213...
......2103.42210001.213.
......1112.43.111233202.
......2.222..311.3..212.
.............2124.4.....
............111.3.......
........................
........................
"""

safe, mines = solve_board(board_str)
print(f"安全格: {len(safe)} 个")
print(f"地雷格: {len(mines)} 个")
show_result(board_str, safe, mines)

安全格: 43 个
地雷格: 54 个
. . * 2 0 1 # . . . . . . . . . . . . . . . . .
. . * 2 0 1 * # . . . . . . . . . . . . . . . .
. . # 1 0 1 3 * . . . . . . . . . . . . . . . .
. . # 1 1 0 2 * # # * # . . . . . . . . . . . .
. # # * 1 0 1 1 1 2 3 # # # * # # # * # # . . .
. . 2 1 1 0 0 0 1 2 * * * 2 2 2 1 1 1 2 2 . . .
. . 2 0 0 0 0 1 2 * 3 3 2 1 2 * 3 1 1 1 * # . .
. * 4 2 1 0 0 2 * 3 1 0 0 0 2 * 3 * 1 2 2 # . .
. . * * 3 2 2 4 * 3 1 0 0 0 1 1 2 1 1 1 * # . .
. . . * * # * * 4 * 2 0 0 0 0 0 0 1 1 2 1 # . .
. . . . . . * 4 3 * 2 0 0 0 0 0 1 2 * 1 1 # . .
. . . . . . * 3 1 2 2 2 1 1 0 0 1 * 2 1 2 * . .
. . . . . * * 2 0 2 * 3 * 1 0 0 1 1 2 1 3 * # .
. . . . . # 2 1 0 3 * 4 2 2 1 0 0 0 1 * 2 1 3 .
. . . . . # 1 1 1 2 * 4 3 * 1 1 1 2 3 3 2 0 2 .
. . . . . # 2 * 2 2 2 * * 3 1 1 * 3 * * 2 1 2 .
. . . . . . . # * # # # * 2 1 2 4 * 4 # . . # .
. . . . . . . . . . . # 1 1 1 * 3 * # # . . . .
. . . . . . . . . . . # # # # # # # . . . . . .
. . . . . . . . . . . . . . . . . . . . . . . .


盘面上没翻开的格子一共 111 个，跑完以后，其中 43 个被判成安全，54 个被判成地雷，都是可以定论的结论。剩下的 14 个，两种假设都试过，哪一种都不矛盾，因此只能暂时搁置，人自己玩的时候同样会卡在这 14 格上，最后仍然要靠猜。

输出里的 `#` 是安全格，`*` 是地雷格，两者合起来就是程序能够确定的部分。

## 代码讲解

### 1. 建立变量

```python
mine = {}
for y, row in enumerate(rows):
    for x, v in enumerate(row):
        if v == '.':
            continue
        for nx, ny in get_neighbors(x, y, h, w):
            if rows[ny][nx] == '.':
                mine[ny, nx] = Bool(f'mine_{ny}_{nx}')
```

`mine` 是一本字典，其中 `(x, y) → Bool 变量` 就是它的键值关系。变量问的是“这一格有没有地雷”，`True` 表示有雷，`False` 表示没有。由于离数字格很远的格子没有任何提示，例如盘面角落的格子可能完全不挨着数字，因此只给数字格旁边的未翻开格子建立变量，否则建出来也无从判断。

### 2. 添加约束条件

```python
around = [mine[loc] for loc in get_neighbors(x, y, h, w) if loc in mine]
if around:
    solver.add(Sum(around) == int(v))
```

拿到数字格 `v` 以后，下面把“周围那些还没翻开的格子，地雷数正好是 `v`”写成一条约束。

`Sum(around)` 是对 `Bool` 变量求和，下面看它为什么能直接给出雷的个数：由于 Z3 把 `True` 当 1、`False` 当 0 加起来，因此其结果就是雷的个数，表示周围埋雷的数量，不需要自己换算。

`around` 变为空，只发生在数字格周围已经全部翻开的时候。这种格没有可加的约束，因此用 `if around:` 直接跳过。

### 3. 用反证法判断

```python
solver.push()
solver.add(var)                   # 假设：这里有地雷
if solver.check() == unsat:       # 矛盾 → 不是地雷
    safe.append((x, y))
solver.pop()
```

| 调用 | 作用 |
|---|---|
| `solver.push()` | 把当前的约束保存一份，之后才好添加临时约束 |
| `solver.add(...)` | 添加这一次的假设 |
| `solver.check()` | 带着假设检查有没有解(`unsat` 就是矛盾) |
| `solver.pop()` | 退回保存下来的状态，把假设扔掉 |

表中最后一行往往最容易被漏掉，漏写 `pop()` 就会让上一格的假设留在求解器里。这样一来，假设会跟着一起审查下一格，答案自然就不可靠了。因此在每次判断之后，`push()` 与 `pop()` 必须成对出现。

## 小结

- 未翻开的格子一旦变成 `Bool` 变量，周围的地雷数就能直接写成 `Sum(...) == 数字`
- 反证法的判断只有两句话：假设有雷却 `unsat`，那就是安全；假设没有雷却 `unsat`，那就是地雷
- 临时假设要用 `push()` 与 `pop()` 前后夹住，不要让假设漏到下一次判断里
- 两种假设都不矛盾的格子，就是真的定不了，扫雷里那种一翻就炸的时刻，根源正在这里